# [2026年4月20日] LLMって何？実演編

本ノートブックでは, HuggingFaceというサイトにあるGeminiモデルを使用して, LLMがどのように動作するか, そして簡単な設定変更によって出力がどう変化するかを体感します.

## LLMのイメージ

大規模言語モデル(large language model; LLM)とは, 膨大なテキストデータに基づいて, **所与の入力に対して尤もらしい(よくある)回答を返す**ように訓練された, 機械学習モデルのことです.

ChatGPTやGemini, Claudeなど「生成AI」の構築と進化において活用されている技術です.

### 言語モデル

- 「次に続く単語」の確率をモデル化しているのが現在の言語モデルです.
    - 例えば「きょうの天気は」と言われて「ピザです」と答えることは常識的あり得ないが「晴れです」と答えることは妥当です.
    - 言い換えれば「きょうの天気は」という文の続きに「ピザ」という単語が来る確率は低いが「晴れ」という単語が来る確率は高いです.
    - こういう「よくある」続きが出てくるように, 人間同様に答え合わせをする(教師あり学習)などして, モデルを訓練する必要があります.

### 大規模であるとは

- 事前に構築した単語全体の集合(語彙)の中で, 所与の文章に続く単語としてどれが「よくあるか」を学習するには, あるあるパターンをモデルに「習得」させる必要があります. これが「機械学習」の「学習」に相当するものです.
- 大規模であるとは, その「習得」いわば「学習」のために使われているデータの量が多いということです. 各所に溢れている膨大な量の文章資源(コーパス)から, 文章のよくあるパターンを「学習」させます. 一般的には**数百万冊分の書籍相当の文章量**から学習しています.
- 今はまだ詳細な学習手法には踏み込まないが, 機械学習で行われる「教師あり学習」や「教師なし学習」そして「強化学習」がよく活用されます.

## 作成者
坪井 一馬
- 横浜国立大学 理工学部 化学・生命系学科 化学EP 4年生
- 化学と情報科学を融合したケモインフォマティクスの研究をしている. 特に, 有機化合物データベースや特許情報を扱う観点でLLMを日々扱う.
- 東京大学松尾・岩澤研究室のLLM講座2025基礎編/応用編を修了済み.
    - 基礎編の修了率47%, 応用編の修了率27%.

## 注意
- 難しいので, 学術的な厳密性や数理的背景には踏み込みません.
- 皆さんの手で動作するには色々準備が必要なので, **ここでは坪井による実演と資料共有にとどめます**.
    - 私の実行では, 有料で高性能なGPUに課金して高速に実施していますが, 無料版であればおそらく30分くらい待機が必要です. 落ちてしまうこともあります.
    - **Lumosに入ってくれたら, 皆さんの手で, 皆さんのPCで色々いじる機会を積極的に設けますのでお楽しみに!**
- 本資料にはGoogle Geminiを使用して構築している部分がありますが, すべて坪井の目を通しており, 誤りがないことを確認済みです.

## 0. APIキーの設定
- 今回はWeb上で公開されているモデルを動かしてみます.
- Geminiモデルを使うには, 事前に取得したAPIキーと呼ばれるものを入力する必要があります. 簡単にいうと**利用者として認定されていることの証明**です.
- 事前に取得の必要がありますが, 今は実演なので坪井のほうで登録済みのものを読ませます.

In [1]:
!pip install -q transformers accelerate

import torch
from transformers import pipeline
from google.colab import userdata

# Hugging Face Tokenの設定
try:
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = None
    print("HF_TOKENが設定されていません。")

def load_hf_model(model_id):
    print(f"{model_id} をロード中...")
    return pipeline(
        "text-generation",
        model=model_id,
        model_kwargs={"torch_dtype": torch.bfloat16},
        device_map="auto",
        token=hf_token
    )

# 2つのモデルを定義
model_lite_id = "google/gemma-2-2b-it"
model_standard_id = "google/gemma-2-9b-it"

print("--- gemma-2-2b-itのモデルをロード ---")
pipe_lite = load_hf_model(model_lite_id)

print("--- gemma-2-9b-itのモデルをロード ---")
pipe_standard = load_hf_model(model_standard_id)

--- gemma-2-2b-itのモデルをロード ---
google/gemma-2-2b-it をロード中...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

## 1. モデルサイズの違い
- まずは, シンプルな質問を投げ, モデルがどのように答えるかを確認しよう.
- 今回は`gemma-2-2b-it`と`gemma-2-9b-it`を比較します.
    - モデル名でわかると思いますが, 9bよりも2bのほうが「軽い」です. ただし, 出力される内容の質は落ちてしまいます.
    - モデルによって**作られ方**(学習資源の量や質, 学習手法)が違っていて, 出力も変わってきます.
    - 軽量ですぐ動かせるモデルよりも, 動かすのに時間のかかるモデルのほうが, モデルの規模が大きいので, より人間らしい回答をしてくれる傾向にあります.

### 1.1 高性能な9b

In [2]:
# 実行するプロンプト
prompt = "横浜国立大学について、大学受験を控えた高校生向けに「簡潔に」説明してください。"

# 9b (Flash相当) モデルで生成
messages = [{"role": "user", "content": prompt}]
outputs = pipe_standard(
    messages,
    max_new_tokens=512,
    do_sample=True,
    temperature=1.0,
)
response_text = outputs[0]["generated_text"][-1]["content"]

print(f"--- プロンプト ---\n{prompt}\n")
print(f"--- LLMの回答 ---\n{response_text}")

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- プロンプト ---
横浜国立大学について、大学受験を控えた高校生向けに「簡潔に」説明してください。

--- LLMの回答 ---
横浜国立大学は、神奈川県横浜市にある、総合的に高い学力が求められる難関大学です。

✅ **強み**
* **研究力**: 各学部で様々な研究が盛んで、優れた教授陣に学びたい人におすすめ。
* **立地**: 横浜市内にあり、アクセス良好で都会的な環境の中で学べる。
* **国際色**: 国内外からの学生が多く、グローバルな環境で学ぶ機会が多い。

❌ **注意点**
* **高い学力が必要**: 入学に必要とされる学力が高いので、長期間の学習と集中力が求められる。
* **競争が激しい**: 多くの受験生が目指す大学なので、合格には相当な努力が必要。

横浜国立大学は高い学力と国際性を重視し、将来を切り開く強い基盤を求める人にとっての魅力的な大学です。



### 1.2 低性能な2b

In [3]:
# 実行するプロンプト
prompt = "横浜国立大学について、大学受験を控えた高校生向けに「簡潔に」説明してください。"

# Hugging FaceのPipeline形式で回答を生成
# messages形式にすることで、チャットモデルとして適切に動作します
messages = [
    {"role": "user", "content": prompt},
]

# 生成実行
outputs = pipe_lite(
    messages,
    max_new_tokens=512,
    do_sample=True,
    temperature=1.0,
)

response_text = outputs[0]["generated_text"][-1]["content"]

print(f"--- プロンプト ---\n{prompt}\n")
print(f"--- LLMの回答 ---\n{response_text}")

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- プロンプト ---
横浜国立大学について、大学受験を控えた高校生向けに「簡潔に」説明してください。

--- LLMの回答 ---
横浜国立大学って、**横浜に住む学生におすすめ**の大学だよ！😊

**特徴**

* **自然豊かなキャンパス**🌳 多くの人が「気持ちがいい」と感じる場所です。海が近くにあって、リラックスできる環境だよね！
* **学部・学科が充実**📖  理系学部も医学部も、人間との関係性で有名な「人間社会」学部も、スポーツ学科も充実してるから、自分に合った学部を選べそうな気がする。
* **実績もすごい！**🎓 大学卒業後、就職や進学率も高く、将来の可能性を広げられるようにサポートしてくれるよ！

**受験難易度**

横浜国立大学は**「難関」**大学！ 
だから、**志望学部・学科をしっかり考えて、勉強計画を立てるのが大切**だよ！ 💪

**まとめ**

横浜国立大学は、**自然と学びが楽しめる**大学。
受験勉強頑張ってね！🍀 

**もっと詳しく知りたい！**

* 横浜国立大学のホームページ: [https://www.ysu.ac.jp/](https://www.ysu.ac.jp/) 
* 大学のホームページで学部・学科を詳しく調べてみよう！


## 2. プロンプトエンジニアリング入門: 役割の指定
- 同様の内容の質問であっても, プロンプトで役割を指定することにより出力が変わることを体感する.

### 2.1 高性能な9b

In [4]:
prompt_specific = """
あなたはベテラン予備校講師で、受験指導の経験が豊富です。
横浜国立大学について、大学受験を控えた高校生向けに「簡潔に」説明してください。
"""

messages = [{"role": "user", "content": prompt_specific}]
outputs = pipe_standard(
    messages,
    max_new_tokens=512,
    do_sample=True,
    temperature=1.0,
)
response_text = outputs[0]["generated_text"][-1]["content"]

print(f"--- 具体的な指示プロンプト ---\n{prompt_specific}\n")
print(f"--- LLMの回答 ---\n{response_text}")

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- 具体的な指示プロンプト ---

あなたはベテラン予備校講師で、受験指導の経験が豊富です。
横浜国立大学について、大学受験を控えた高校生向けに「簡潔に」説明してください。


--- LLMの回答 ---
横浜国立大学は、神奈川県横浜市にある、文系・理系の両方でレベルの高い総合大学です。 

**特徴としては**

* **理系の強み:** 工学部、理学部は特に有名で、研究も盛ん。
* **人間科学系が充実:** 教育学部をはじめ、心理学系や社会学系など、社会貢献につながる分野も強み。
* **国際色豊か:**  多くの留学生が在学し、英語教育にも力が入っている。Campusも都会的で施設は充実しています。
* **就職力が高い:** 企業からも好まれる大学で、特に地元企業との関わりも深く。

横浜国立大学を目指すなら、**得意教科を伸ばすだけでなく、幅広い興味関心を持ち、活動しておくこと**が重要です。


大学選びは人生のターニングポイントです。ご自身の興味や将来像をよく考え、ぜひ横浜国立大学で学びたいと思った場合は、頑張って下さい！  




### 2.2 低性能な2b

In [5]:
# 2bモデルでの実行
prompt_specific = """
あなたはベテラン予備校講師で、受験指導の経験が豊富です。
横浜国立大学について、大学受験を控えた高校生向けに「簡潔に」説明してください。
"""

messages = [{"role": "user", "content": prompt_specific}]
outputs = pipe_lite(
    messages,
    max_new_tokens=512,
    do_sample=True,
    temperature=1.0,
)
response_text = outputs[0]["generated_text"][-1]["content"]

print(f"--- 具体的な指示プロンプト ---\n{prompt_specific}\n")
print(f"--- LLMの回答 ---\n{response_text}")

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- 具体的な指示プロンプト ---

あなたはベテラン予備校講師で、受験指導の経験が豊富です。
横浜国立大学について、大学受験を控えた高校生向けに「簡潔に」説明してください。


--- LLMの回答 ---
横浜国立大学って、横浜って近くて有名な大学！ 😉

**一言で言い表したら、** **「学業と社会貢献に優れている大学」** です。

**魅力ポイントを３つに絞ります:**

1. **難しいけど、挑戦しやすく、充実した学習環境:** 進路や成績で悩んでしまう生徒にも、大学生活を楽しみながら勉強できる環境です。
2. **多彩な学部・学科:**  STEM（科学・理工学）、人間教育、国際関係など、幅広い分野があります。自分に合った学部を見つけ出せる！
3. **充実したキャンパスライフ:**  横浜ならではの海辺、緑豊かなキャンパス。スポーツなど、充実したサークル活動も魅力です。

**就職後のキャリアパスも注目！**
  横浜国立大学は、多くの人が就職活動をして、将来を築いていくことができる、大学です。

**まとめ:**  
横浜国立大学は、大学受験を控えた高校生にとって、**「挑戦と学び、社会貢献」** を目指す上で最適な選択肢と言えるでしょう。 


**補足:**
横浜国立大学についてもっと知りたい場合は、公式ホームページや受験情報サイトで情報収集しましょう。 
応援しています！ 😊 🎉 



## 3. 字数制御って難しい！
- 文字数を指定して喋らせましょう.
- 日本語で「300字」として制限をつけて動かしてみるけど...あれ？

### 3.1 高性能な9b

In [6]:
prompt = """
横浜国立大学について、以下の制約を厳守して簡潔に説明してください。

1. 日本語で「簡潔に」説明すること。
2. 句読点を含めて、合計で【ちょうど300文字】にすること。
3. 解説の最後に、それまでに出力した文章の合計文字数を答える。
"""

# 9bモデルで実行
messages = [{"role": "user", "content": prompt}]
outputs = pipe_standard(
    messages,
    max_new_tokens=512,
    do_sample=True,
    temperature=1.0)
content = outputs[0]["generated_text"][-1]["content"]

# 文字数カウント
actual_count = len(content.replace('\n', '').replace(' ', ''))

print(f"--- LLMの回答 ---\n{content}\n")
print(f"--- 判定 ---")
print(f"指示した文字数: 300文字")
print(f"実際の文字数: {actual_count}文字 (改行・空白除く)")
print(f"誤差: {actual_count - 300}文字")

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- LLMの回答 ---
横浜国立大学は、神奈川県横浜市にある国立大学です。1939年に創立され、文系の学部と理系の学部から広く多様な研究分野をカバーする、総合型の国立大学としての顔を持っています。特に、教育や工学、経営、人文科学などの分野で高い評価を得ています。

キャンパスは横浜市内の郊外に位置し、緑豊かな環境の中で学ぶことができます。国際的な交流も多く活発に行われており、外国からの学生や研究者も多く在籍しています。横浜国立大学は、豊かな歴史と伝統を基盤に、未来を担う人材を育成するための教育・研究に力を入れています。

(出力文字数: 174) 




--- 判定 ---
指示した文字数: 300文字
実際の文字数: 259文字 (改行・空白除く)
誤差: -41文字


### 3.2 低性能な2b

In [7]:
# 2bモデルで実行
messages = [{"role": "user", "content": prompt}]
outputs = pipe_standard(
    messages,
    max_new_tokens=512,
    do_sample=True,
    temperature=1.0)
content = outputs[0]["generated_text"][-1]["content"]

actual_count = len(content.replace('\n', '').replace(' ', ''))

print(f"--- LLMの回答 ---\n{content}\n")
print(f"--- 判定 ---")
print(f"指示した文字数: 300文字")
print(f"実際の文字数: {actual_count}文字 (改行・空白除く)")
print(f"誤差: {actual_count - 300}文字")

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- LLMの回答 ---
横浜国立大学は、神奈川県横浜市に位置する国立大学です。

1939年に国立横浜高等工学校として創立され、その後、大学の設立を経て現在に至ります。広い敷地に、理学、工学、教育学、経済学など、多彩な学部・研究科を擁しています。

横浜という都市の魅力を活かし、国際交流にも力を入れており、国内外の学生が学ぶ環境を提供しています。

教育研究では、高度な技術や知識の習得だけでなく、国際的な視野や問題解決能力を育むことを重視し、将来を担う多様な人材を育成しています。

【227文字】 




--- 判定 ---
指示した文字数: 300文字
実際の文字数: 231文字 (改行・空白除く)
誤差: -69文字


## 4. パラメータによる変化
- こんなのGeminiの画面でやるのと変わらないと思ったアナタへ
    - LLMエンジニアリングの世界では, **モデルから所望の回答を引き出す**ために. **設定値を調整する**とか, **他のモデルと組み合わせる**ことがあります. 入力を変えるだけではありません.
    - 他のモデルとの組み合わせ(生成結果の評価)や, 連続的な出力で過去の出力内容を保存して反映するなどの手法は難しいので, 本日は扱いませんが, そういうコードベースでの**プログラミングによる工夫もできる**というのは知っておいてほしいです.
- LLMの振る舞いを決める指標として, 様々な**数値的パラメータ**があります.
    - どのくらいの長さまで生成をさせるか, 途中まで候補を留め置いておくか否かなど.
    - ここでは, 回答の「いい加減さ」を制御する温度パラメータ`temperature`を調整してみます. 温度パラメータは0.0から2.0の範囲で指定可能です.
        - **低い値 (0.1程度)**: 常に最も確率の高い言葉を選び, 論理的で安定した回答になります. ビジネス的には最適だがつまらない？
        - **中程度の値 (1.0程度)**: 確率的に尤もらしいものを出します. 事実ベースで間違ったり, おかしなものを生成することもあります. ここまでの実演では1.0にしていました.
        - **高い値 (2.0程度)**: 生成が崩壊することもあります. 面白おかしい文章を生成したいならば有効ではありますが...
- 私の事前テストで不適切な内容が出てしまったため「不適切な内容は出力しないこと」というプロンプトを明示的に入れます. これだけでもかなり効きます.

In [8]:
def test_temp_hf(pipe, temp_value):
    creative_prompt = "あなたは日本語を話すピザ職人です。売れそうなピザのアイディアを3つ出してください。不適切な内容は出力しないこと。"
    messages = [{"role": "user", "content": creative_prompt}]

    outputs = pipe(
        messages,
        max_new_tokens=512,
        do_sample=True if temp_value > 0 else False,
        temperature=temp_value if temp_value > 0 else None
    )

    print(f"=== Temperature: {temp_value} ===")
    print(outputs[0]["generated_text"][-1]["content"])

### 4.1 高性能な9b

In [9]:
print("堅実な回答 (9b)")
test_temp_hf(pipe_standard, 0.1)

print("-" * 30)

print("創造的な回答 (9b)")
test_temp_hf(pipe_standard, 1.0)

print("-" * 30)

print("生成が崩壊するかも (9b)")
test_temp_hf(pipe_standard, 2.0)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


堅実な回答 (9b)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Temperature: 0.1 ===
はい、かしこまりました！

売れそうなピザのアイデアですね！ 

1. **和風照り焼きチキンピザ:**  甘辛い照り焼きチキンと、ネギ、マヨネーズをトッピングした和風テイストのピザ。日本の味をピザで楽しめる、新しい感覚です！
2. **和風サーモンとアボカドピザ:**  新鮮なサーモンとアボカド、クリームチーズを組み合わせた、ヘルシーで贅沢なピザ。和風ドレッシングをかけるとさらに美味しくなりますよ！
3. **きのこたっぷりチーズピザ:**  きのこの種類を色々使い、チーズと相性抜群の組み合わせに仕上げた、きのこ好きにはたまらないピザ！ 

いかがでしょうか？ どれも人気が出そうな気がします！ 🍕

------------------------------
創造的な回答 (9b)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Temperature: 1.0 ===
よ！おいしそうピザ探してるんか？

俺がおすすめする3つのピザは、これだ！

1. **和風カルボナーラピザ**:  ピザ生地に和風カルボナーラをたっぷり乗せたピザ！  濃厚なカルボナーラと、ほんのり香る醤油の合わせで、新しい味覚を味わえるんや！
2. **照り焼きチキンとピクルスピザ**:  甘辛い照り焼きチキンと、ピリ辛のピクルスのコントラストがたまらない一品！  ちょっと冒険したい人におすすめだ！
3. **ツナとクリームチーズのメカジ巻ピザ**:  懐かしい味を、ピザで楽しみたい人にぴったり！  ツナとクリームチーズの合う組み合わせは最強！

どれもお腹も心も満たしてくれる味なんだ！  ぜひお試しあれ！🍕




------------------------------
生成が崩壊するかも (9b)
=== Temperature: 2.0 ===
やあ！ お馴染みのピザ屋マスターだよん！  今週注目の新しいピザができたから、お客さんの口の中で飛び散らせ、売れ面上にするために３つの最高のイディアを送ってくるぜ！

**斬新で旨さの爆発を Promise!**

 **①炙りの香りに食欲刺激  チチリアントのロマンチック・ポポロ!**
     - ネーミングをつけたロマンチックな一品ね! 地元こだわりのおろしポコロにモントチッチ焼きチーズ、唐辛子の効いたハーブソースを合わせてアチッチチ 🌶️ あたり外れのなさ〜いね! 売れ残ったって…こわかったけど美味しい!

 **②ピリ旨コラボで中毒性もアリ スウィート🌶  モントチセリ!** をりっつする!!
    - ピなしモヤ、トマトマリレードな具材たちにピリ辛のハラペーニョを加えて、少し甘味もトッピングしたら 🌶🌮 スイスード！ 食べ続けてしまうやつだ...。売りのキャッチコピーは "あと一本食べた後？ やっぱり、130キロカロリーじゃない、22キロは？" 

 **③とろ cheesy 極 🧀  モカ・オズン! (和風モダンピザ!)**:
      - ちょっと高級路線へ向けさせてもろ〜。抹茶に溶かしたクリーム、濃厚ガーリックピザに加えている  、ブラックペッパーの効かせる。お酒だって合えばええやろ！  



どうだい？ 今日の出来上がったピザとワイン

### 4.2 低性能な2b

In [10]:
print("堅実な回答 (2b)")
test_temp_hf(pipe_lite, 0.1)

print("-" * 30)

print("創造的な回答 (2b)")
test_temp_hf(pipe_lite, 1.0)

print("-" * 30)

print("生成が崩壊するかも (2b)")
test_temp_hf(pipe_lite, 2.0)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


堅実な回答 (2b)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Temperature: 0.1 ===
こんにちは！ピザ職人、[あなたの名前]です。

売れそうなピザのアイデアを3つ、お伝えします！

1. **「秋の味覚」ピザ:**  秋の味覚をたっぷり使ったピザです。ternut squash、きのこ、ベーコン、 sage、クリームチーズなど、旬の食材を組み合わせ、濃厚な味わいが特徴です。
2. **「チーズとハーブ」ピザ:**  定番のチーズとハーブを組み合わせたピザです。モッツァレラチーズ、ゴルゴンゾーラチーズ、パルメザンチーズ、ローズマリー、タイム、バジルなど、様々なハーブとチーズを組み合わせ、爽やかな味わいが特徴です。
3. **「和風」ピザ:**  和風食材を使ったピザです。鶏肉、野菜、だし汁、醤油、みりん、砂糖などを使い、和風テイストのピザです。

これらのピザは、季節や流行に合わせて、アレンジを加えることもできます。 

何かご希望があれば、遠慮なく聞いてください！ 🍕 

------------------------------
創造的な回答 (2b)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Temperature: 1.0 ===
こんにちは！ピザ職人です。大人気ピザのアイデアを3つご紹介します！

1. **和風ピザ：**  
   - 豚肉のソテー、きゅうり、ねぎ、甘味噌、そして焼き豆腐と、少しピリ辛のしょうゆ風味を組み合わせた和風なピザは、人気の高い味です！
2. **秋の味覚ピザ：** 
   - 旬の栗、きのこ、そしてモッツァレラチーズがたっぷり入った、秋の味覚を閉じ込めるピザは、秋ならではの美味しさで人気です！
3. **チーズと梅のピザ：**  
   - 濃厚なトマトソースと、とろけるようなチーズ、そして甘酸っぱく香り立つ梅の塩漬けが織りなす、意外にもマッチする組み合わせはいかがでしょうか？ 




これらのアイデアをベースに、あなたのオリジナルのピザも楽しんでください！
------------------------------
生成が崩壊するかも (2b)
=== Temperature: 2.0 ===
かしこם！新作ピザのごアイデア、たくさん作ってこようか！ 😎 

日本でも人気なのは 、、
 1. **秋の味覚🍂  - キノコとベーコンのピザ**: サックリ系の生地にお好きな野菜と自家製ベーコンを焼けば、まるで秋の夜空を味わえる！ 🍁✨ お酒もぴったりですね！ 
  
 2. **冬から春を感じる！ ✨ - 甘辛 Chili と鶏肉とアスパ拉のピザ***: これは寒い時期を吹き飛ばしてくれます  ！ 一種類しかない「春は甘いもの！」！ 🍫🥚🐔🌿 でアレンジも可能ですよ 😎 。

3.  **夏の大自然🌿💖 **のピザ は- レモンシップとチーズの塩レモン ピzza**:  甘酸っぱい  風味のレモンシップと甘みたのチーズの香りが、夏の夜の香りの旅へ連れてってくれます。 🍋🧀🏖


どうですね？ どんなピザがヒットしますか！ 😄🇮🇹


## 5. 保存
- 何か実行をしたら, それを再現できるように保存をしておくことは重要です.
- 皆さんに共有をするため, GitHub Gistというサイトへのアップロード, HTMLとしての出力を用意します. これは事務的なコードです.

In [13]:
import json
from google.colab import _message
from google.colab import userdata
import requests

# --- GitHub Gist & HTML出力設定 ---
save_name = "260420_What_is_LLM_test"
save_name_gist = "260420_What_is_LLM_test.ipynb"
# ----------

# 1. 現在のノートブックのJSONデータを取得
notebook_json_raw = _message.blocking_request('get_ipynb', request='', timeout_sec=5)
ipynb_dict_raw = notebook_json_raw['ipynb']

# --- GitHub Gistへの保存 ---
print("---> GitHub Gistへの保存を開始します --->")

try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception as e:
    github_token = None
    print(f"GitHub TOKENの取得に失敗しました: {e}")
    print("ColabのSecretsに 'GITHUB_TOKEN' を設定してください。")

if github_token is None:
    print("GitHub TOKENが設定されていないため、Gistの作成をスキップします。")
else:
    filename_gist = save_name_gist
    description_gist = "LLMって何かを解説する"
    is_public_gist_str = 'yes'
    is_public_gist = True if is_public_gist_str == 'yes' else False

    url_gist = 'https://api.github.com/gists'
    headers_gist = {
        'Authorization': f'token {github_token}',
        'Accept': 'application/vnd.github.v3+json'
    }

    ipynb_dict_for_gist = json.loads(json.dumps(ipynb_dict_raw)) # Deep copy

    if 'metadata' not in ipynb_dict_for_gist:
        ipynb_dict_for_gist['metadata'] = {}
    if 'widgets' not in ipynb_dict_for_gist['metadata']:
        ipynb_dict_for_gist['metadata']['widgets'] = {}

    if "application/vnd.jupyter.widget-state+json" not in ipynb_dict_for_gist['metadata']['widgets'] or not isinstance(ipynb_dict_for_gist['metadata']['widgets']["application/vnd.jupyter.widget-state+json"], dict):
        ipynb_dict_for_gist['metadata']['widgets']["application/vnd.jupyter.widget-state+json"] = {}

    ipynb_dict_for_gist['metadata']['widgets']["application/vnd.jupyter.widget-state+json"]["state"] = {}

    notebook_content_str_gist = json.dumps(ipynb_dict_for_gist, ensure_ascii=False, indent=4)

    data_gist = {
        'description': description_gist,
        'public': is_public_gist,
        'files': {
            filename_gist: {
                'content': notebook_content_str_gist
            }
        }
    }

    print("\nGistを作成中...")
    try:
        response_gist = requests.post(url_gist, headers=headers_gist, data=json.dumps(data_gist))
        response_gist.raise_for_status()

        gist_data = response_gist.json()
        print(f"Gistが正常に作成されました！\nURL: {gist_data['html_url']}")
    except requests.exceptions.HTTPError as err:
        print(f"HTTPエラーが発生しました: {err}")
        print(f"レスポンス: {response_gist.text}")
    except Exception as err:
        print(f"Gistの作成中にエラーが発生しました: {err}")

---> GitHub Gistへの保存を開始します --->

Gistを作成中...
Gistが正常に作成されました！
URL: https://gist.github.com/Tsuboi-coder/8037151d1216ff123db8b9b2c7e9bd59


In [14]:
print("\n--- HTML出力を開始します ---> ")

ipynb_dict_for_html = json.loads(json.dumps(ipynb_dict_raw)) # ディープコピーを作成

# KeyError: 'state' 回避のため、メタデータからwidgets情報をクリア
if 'metadata' in ipynb_dict_for_html:
    if 'widgets' in ipynb_dict_for_html['metadata']:
        del ipynb_dict_for_html['metadata']['widgets'] # widgetsキー自体を削除

# 指定した名前でipynbファイルとして保存
ipynb_filename_html = f'{save_name}.ipynb'
with open(ipynb_filename_html, 'w', encoding='utf-8') as f:
    json.dump(ipynb_dict_for_html, f, ensure_ascii=False, indent=4)

# 保存したipynbファイルをHTMLに変換
!jupyter nbconvert --to html {ipynb_filename_html}

print(f'\n--- 完了 ---\nHTML出力ファイル: {save_name}.html が作成されました。左側のファイルメニューからダウンロードしてください。')


--- HTML出力を開始します ---> 
[NbConvertApp] Converting notebook 260420_What_is_LLM_test.ipynb to html
[NbConvertApp] Writing 353635 bytes to 260420_What_is_LLM_test.html

--- 完了 ---
HTML出力ファイル: 260420_What_is_LLM_test.html が作成されました。左側のファイルメニューからダウンロードしてください。
